Data Cleaning: Aquifer Coverage cropped to mine areal extent

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt

In [70]:
aquifers = gpd.read_file('../../data/geojson/aquifers-4326.geojson')

In [ ]:
# correct values by mapping incorrect to correct vals
character_corrections = {
    "Low productive aquifer": "Low productivity aquifer",
    "Modeately productivity aquifer": "Moderately productive aquifer",
}
# apply corrections
aquifers['CHARACTER'] = aquifers['CHARACTER'].replace(character_corrections)

# rename null values in FLOW_MECHA column
aquifers.loc[aquifers['FLOW_MECHA'].isna(), 'FLOW_MECHA'] = 'No flow mechanism'



,OBJECTID,ROCK_UNIT,CLASS,CHARACTER,FLOW_MECHA,SUMMARY,VERSION,geometry
0,1.0,GAULT FORMATION,3,Rocks with essentially no groundwater,No Flow Mechanism,Low permeability clays up to 100 m thick separ...,HydrogeologyUK_IoM_v5,"MULTIPOLYGON (((-2.35032 51.1259, -2.35033 51...."
1,2.0,GAULT FORMATION,3,Rocks with essentially no groundwater,No Flow Mechanism,Low permeability clays up to 100 m thick separ...,HydrogeologyUK_IoM_v5,"MULTIPOLYGON (((-2.35171 51.13027, -2.35409 51..."
2,3.0,GAULT FORMATION,3,Rocks with essentially no groundwater,No Flow Mechanism,Low permeability clays up to 100 m thick separ...,HydrogeologyUK_IoM_v5,"MULTIPOLYGON (((-2.34307 51.13774, -2.34097 51..."
3,4.0,GAULT FORMATION,3,Rocks with essentially no groundwater,No Flow Mechanism,Low permeability clays up to 100 m thick separ...,HydrogeologyUK_IoM_v5,"MULTIPOLYGON (((-2.33657 51.14115, -2.34014 51..."
4,5.0,GAULT FORMATION,3,Rocks with essentially no groundwater,No Flow Mechanism,Low permeability clays up to 100 m thick separ...,HydrogeologyUK_IoM_v5,"MULTIPOLYGON (((-2.28896 51.17141, -2.2897 51...."
...,...,...,...,...,...,...,...,...
13777,13780.0,GREY CHALK SUBGROUP,2A,Highly productive aquifer,Flow is virtually all through fractures and ot...,Marly Chalk aquifer that can yield up to 5 L/s...,HydrogeologyUK_IoM_v5,"MULTIPOLYGON (((-0.1047 51.26515, -0.10476 51...."
13778,13781.0,FELL SANDSTONE GROUP,1B,Moderately productive aquifer,Significant intergranular flow,Locally important aquifer up to 300 m thick wi...,HydrogeologyUK_IoM_v5,"MULTIPOLYGON (((-2.02017 55.77472, -2.0196 55...."
13779,13782.0,UPPER GREENSAND FORMATION,1B,Moderately productive aquifer,Significant intergranular flow,Glauconitic sands yielding up to 25 L/s and of...,HydrogeologyUK_IoM_v5,"MULTIPOLYGON (((-2.99223 50.8445, -2.99223 50...."
13780,13783.0,UPPER GREENSAND FORMATION,1B,Moderately productive aquifer,Significant intergranular flow,Glauconitic sands yielding up to 25 L/s and of...,HydrogeologyUK_IoM_v5,"MULTIPOLYGON (((-2.99223 50.8445, -2.99421 50...."


In [84]:
# crop aquifers by proximity to mine boundaries
mines = gpd.read_file('../../data/geojson/coalfield-extent-4326.geojson')

mines_27700 = mines.to_crs(epsg=27700)
aquifers_27700 = aquifers.to_crs(epsg=27700)

# buffer to allow for nearby aquifers (range: 5km)
mine_buffer = mines_27700.buffer(5000)

# spatial join to keep only aquifers within area
mine_buffer_df = gpd.GeoDataFrame(geometry=mine_buffer, crs=mines_27700.crs)
aquifers_cropped = gpd.sjoin(aquifers_27700, mine_buffer_df, how='inner', predicate='intersects')

# drop mine-related columns in df
aquifers_cropped = aquifers_cropped[aquifers.columns]
aquifers_cropped = aquifers_cropped.drop_duplicates()

aquifers_cropped
# aquifers_cropped = aquifers_cropped.to_crs(epsg=4326)

# aquifers

,OBJECTID,ROCK_UNIT,CLASS,CHARACTER,FLOW_MECHA,SUMMARY,VERSION,geometry
10,11.0,GAULT FORMATION,3,Rocks with essentially no groundwater,No Flow Mechanism,Low permeability clays up to 100 m thick separ...,HydrogeologyUK_IoM_v5,"MULTIPOLYGON (((378707.295 144369.945, 378386...."
191,192.0,UPPER GREENSAND FORMATION,1B,Moderately productive aquifer,Significant intergranular flow,Glauconitic sands yielding up to 25 L/s and of...,HydrogeologyUK_IoM_v5,"MULTIPOLYGON (((373558.127 142866.411, 373467...."
268,269.0,GAULT FORMATION,3,Rocks with essentially no groundwater,No Flow Mechanism,Low permeability clays up to 100 m thick separ...,HydrogeologyUK_IoM_v5,"MULTIPOLYGON (((373639.604 143254.333, 373684...."
270,271.0,GAULT FORMATION,3,Rocks with essentially no groundwater,No Flow Mechanism,Low permeability clays up to 100 m thick separ...,HydrogeologyUK_IoM_v5,"MULTIPOLYGON (((400000.492 154509.511, 399380...."
318,319.0,GAULT FORMATION,3,Rocks with essentially no groundwater,No Flow Mechanism,Low permeability clays up to 100 m thick separ...,HydrogeologyUK_IoM_v5,"MULTIPOLYGON (((622082.429 135795.203, 622037...."
...,...,...,...,...,...,...,...,...
13763,13766.0,BRIDPORT SAND FORMATION,1B,Moderately productive aquifer,Significant intergranular flow,Local sand and silt aquifer often in hydraulic...,HydrogeologyUK_IoM_v5,"MULTIPOLYGON (((355674.682 167279.815, 355786...."
13764,13767.0,BRIDPORT SAND FORMATION,1B,Moderately productive aquifer,Significant intergranular flow,Local sand and silt aquifer often in hydraulic...,HydrogeologyUK_IoM_v5,"MULTIPOLYGON (((376600.351 171497.704, 376604...."
13765,13768.0,BRIDPORT SAND FORMATION,1B,Moderately productive aquifer,Significant intergranular flow,Local sand and silt aquifer often in hydraulic...,HydrogeologyUK_IoM_v5,"MULTIPOLYGON (((383797.254 174335.9, 383773.24..."
13766,13769.0,BRIDPORT SAND FORMATION,1B,Moderately productive aquifer,Significant intergranular flow,Local sand and silt aquifer often in hydraulic...,HydrogeologyUK_IoM_v5,"MULTIPOLYGON (((379850.264 198595.353, 379855...."


In [85]:
aquifers_cropped.to_file('../../data/geojson/aquifers-cropped-4326.geojson')

In [72]:
aquifers[['FLOW_MECHA', 'CHARACTER']].groupby(['FLOW_MECHA', 'CHARACTER']).value_counts()

FLOW_MECHA                                                         CHARACTER                            
Flow is virtually all through fractures and other discontinuities  Highly productive aquifer                 575
                                                                   Low productivity aquifer                 7464
                                                                   Moderately productive aquifer            2451
No Flow Mechanism                                                  Rocks with essentially no groundwater    1278
Significant intergranular flow                                     Highly productive aquifer                 302
                                                                   Low productivity aquifer                  184
                                                                   Moderately productive aquifer            1528
Name: count, dtype: int64

In [75]:
aquifers[['SUMMARY', 'FLOW_MECHA', 'CHARACTER']].groupby(['FLOW_MECHA', 'SUMMARY']).value_counts()

FLOW_MECHA                                                         SUMMARY                                                                                                                                                                                           CHARACTER                    
Flow is virtually all through fractures and other discontinuities  Aquifer not exploited as low yielding and only occupies small areas.                                                                                                                              Low productivity aquifer           2
                                                                   Breccia-conglomerate and sandstone yield small amounts of groundwater.                                                                                                                            Low productivity aquifer          10
                                                                   Conglomerate, low productivity fractured aquif